# Use Case 24 — Manufacturing Quality Insight

**Important:** The PDF lists required CSV names but does not supply actual data files. This package contains synthetic demonstration CSVs; replace them before reporting real findings.

## 1. Load libraries and datasets

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA

ROOT = Path('..')
raw = ROOT / 'data' / 'raw'
production = pd.read_csv(raw/'production.csv')
machines = pd.read_csv(raw/'machines.csv')
process = pd.read_csv(raw/'process.csv')
materials = pd.read_csv(raw/'materials.csv')
inspection = pd.read_csv(raw/'inspection.csv')
for name, df in {'production':production,'machines':machines,'process':process,'materials':materials,'inspection':inspection}.items():
    print(name, df.shape)
    display(df.head())

## 2. Data quality audit

In [ ]:
datasets = {'production':production,'machines':machines,'process':process,'materials':materials,'inspection':inspection}
for name, df in datasets.items():
    print(f'\\n{name}: dtypes')
    display(df.dtypes.to_frame('dtype'))
    print('Missing values:')
    display(df.isna().sum().to_frame('missing'))
    print('Duplicate rows:', df.duplicated().sum())
print('Duplicate production batches:', production['batch'].duplicated().sum())

## 3. Clean and combine at batch level

In [ ]:
production = production.drop_duplicates('batch', keep='first')
label_map = {'surface crack':'Surface Crack','dimensional error':'Dimensional Error','porosity':'Porosity','burr':'Burr','weld defect':'Weld Defect','no defect':'No Defect'}
inspection['defect_type'] = inspection['defect_type'].astype(str).str.strip().str.lower().map(label_map).fillna(inspection['defect_type'])
process.loc[~process['temperature'].between(120,260), 'temperature'] = np.nan
process.loc[~process['pressure'].between(30,120), 'pressure'] = np.nan
process.loc[~process['speed'].between(40,200), 'speed'] = np.nan
clean = production.merge(machines,on='batch',how='left').merge(process,on='batch',how='left').merge(materials,on='batch',how='left').merge(inspection,on='batch',how='left')
clean['material_match'] = clean['supplier'].notna()
for col in ['temperature','pressure','speed']:
    clean[col] = clean[col].fillna(clean[col].median())
clean['defect_rate_pct'] = clean['defect_count']/clean['quantity']*100
clean['high_defect_flag'] = (clean['defect_rate_pct'] >= clean['defect_rate_pct'].quantile(.75)).astype(int)
clean['machine_hours_band'] = pd.cut(clean['operating_hours'],[0,3000,6000,9000,12000],labels=['0-3k','3k-6k','6k-9k','9+'],include_lowest=True)
clean['process_deviation_score'] = np.sqrt(((clean['temperature']-185)/12)**2+((clean['pressure']-72)/7)**2+((clean['speed']-115)/14)**2)
clean.to_csv('../data/processed/cleaned_manufacturing_quality.csv',index=False)
clean.head()

## 4. EDA: line, defect type, shift, process, supplier, machine hours

In [ ]:
line = clean.groupby('line').agg(batches=('batch','count'),units=('quantity','sum'),defects=('defect_count','sum'))
line['defect_rate_pct'] = line['defects']/line['units']*100
display(line.sort_values('defect_rate_pct',ascending=False))
line['defect_rate_pct'].sort_values().plot(kind='barh',title='Defect rate by line',ylabel='Line',xlabel='Defect rate (%)'); plt.tight_layout(); plt.show()

display(clean.groupby('defect_type')['defect_count'].sum().sort_values(ascending=False).to_frame('defects'))
clean.groupby('defect_type')['defect_count'].sum().sort_values(ascending=False).plot(kind='bar',title='Defect types'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.show()
shift = clean.groupby('shift').agg(units=('quantity','sum'),defects=('defect_count','sum'))
shift['defect_rate_pct'] = shift['defects']/shift['units']*100
display(shift)
clean.groupby('quality_status')[['temperature','pressure','speed','operating_hours','defect_rate_pct']].mean().round(2)

## 5. Supplier and operating-hours comparisons

In [ ]:
supplier = clean.groupby('supplier',dropna=False).agg(units=('quantity','sum'),defects=('defect_count','sum'))
supplier['defect_rate_pct'] = supplier['defects']/supplier['units']*100
display(supplier.sort_values('defect_rate_pct',ascending=False))
hours = clean.groupby('machine_hours_band',observed=True).agg(units=('quantity','sum'),defects=('defect_count','sum'))
hours['defect_rate_pct'] = hours['defects']/hours['units']*100
display(hours)
clean[['temperature','pressure','speed','operating_hours','defect_count','defect_rate_pct']].corr().round(2)

## 6. Feature engineering, encoding, selection and PCA

In [ ]:
features = clean[['line','shift','machine','age_band','operating_hours','temperature','pressure','speed','supplier','material_grade','quantity','process_deviation_score']]
X = pd.get_dummies(features,drop_first=True)
y = clean['defect_rate_pct']
selector = SelectKBest(f_regression,k=min(10,X.shape[1])).fit(X,y)
feature_scores = pd.DataFrame({'feature':X.columns,'f_score':selector.scores_}).sort_values('f_score',ascending=False)
display(feature_scores.head(15))
num_cols = ['temperature','pressure','speed','operating_hours','quantity','process_deviation_score']
scaled = StandardScaler().fit_transform(clean[num_cols])
pca = PCA().fit(scaled)
pd.DataFrame({'component':[f'PC{i+1}' for i in range(len(num_cols))],'variance':pca.explained_variance_ratio_,'cumulative':np.cumsum(pca.explained_variance_ratio_)})

## 7. Save cleaned output and interpretation

Do not interpret synthetic patterns as actual PrimeForge evidence. Repeat all analysis on the official datasets.